In [1]:
%pwd

'/home/phil/Coding/thesis-code/arco/notebooks'

In [2]:
%cd ..

/home/phil/Coding/thesis-code/arco


In [3]:
# Load OPENAI API KEY from the keychain
import os
import subprocess

os.environ["OPENAI_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openai"], text=True
)
os.environ["OPENROUTER_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openrouter"], text=True
)

In [4]:
from typing import override

from langchain_core.language_models import BaseChatModel

from arco.cli.viz import display_workflow_notebook
from arco.core import Agent, AgentType, Config, State
from arco.core.llm_tools import CoTRefiner
from arco.workflows import Workflow
from arco.workflows.graph import END, Graph

In [17]:
class CodingAgent(Agent):
    CODING_PROMPT = """You're a coding agent.\nGiven a question, firstly reply with a short reasoning on what you would like to implement, then report the implementation and finally comment on specific details on that implementation\n\n##QUESTION\n{prompt}"""

    @override
    def core(self, state: State, llm: BaseChatModel | CoTRefiner) -> State:
        output = self.invoke_llm(
            llm, CodingAgent.CODING_PROMPT.format(prompt=state.prompt)
        )
        return self.answer(
            state,
            message="Code has been generated",
            output={"code": output.text},
            logprobs=output.logprobs,
        )


class ScoringAgent(Agent):
    SCORING_PROMPT = """You're a scoring agent.\nGiven code, firstly reply with a short reasoning on how you would like to score the input as a series of well-defined criteria, then report the scoring values for each criteria. Format the scoring as a JSON\n\n##INPUT\n{last_output}"""

    @override
    def core(self, state: State, llm: BaseChatModel | CoTRefiner) -> State:
        last_output: str = state.get_last_answer().agent_output
        output = self.invoke_llm(
            llm, ScoringAgent.SCORING_PROMPT.format(last_output=last_output)
        )
        return self.answer(
            state,
            message=f"The evaluation is : {output.text}",
            logprobs=output.logprobs,
        )


class MyWorkflow(Workflow):
    workflow_id: str = "my_workflow"

    @override
    def initialize(self, config: Config, graph: Graph):
        coding_agent = CodingAgent()
        scoring_agent = ScoringAgent()

        graph.add_node(coding_agent)
        graph.add_node(scoring_agent)

        graph.set_entry_point(coding_agent)
        graph.add_edge(coding_agent, scoring_agent)
        graph.add_edge(scoring_agent, END)


workflow = MyWorkflow()
print(workflow)

  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
+-------------+  
| CodingAgent |  
+-------------+  
        *        
        *        
        *        
+--------------+ 
| ScoringAgent | 
+--------------+ 
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [19]:
final_state: State = display_workflow_notebook(workflow.stream(), verbose=True)

🚀 Run `scoring-nexus-255`

⏳ Checking 1 model(s)…

▶ **CodingAgent**

✅ **CodingAgent**

▶ **CodingAgent**

▶ **ScoringAgent**

✅ **ScoringAgent**

▶ **ScoringAgent**

✅ Completed — total time 5.45s

In [20]:
final_state.get_last_answer(AgentType.CODINGAGENT).agent_output["code"]

"It seems that your question is incomplete. Please provide the specific coding question or problem you'd like assistance with, and I'll be happy to help!"

In [21]:
final_state.get_last_answer(AgentType.SCORINGAGENT).message

'The evaluation is : To score the provided code, I will evaluate it based on the following criteria:\n\n1. **Clarity**: How clearly does the code communicate its intent?\n2. **Completeness**: Does the code address the user\'s request adequately?\n3. **Engagement**: Does the code encourage further interaction or provide a pathway for the user to continue the conversation?\n4. **Politeness**: Is the tone of the response respectful and courteous?\n\nNow, I will assign scores for each criterion on a scale from 0 to 10, where 0 is the lowest and 10 is the highest.\n\n### Scoring:\n\n- **Clarity**: 8 (The message is clear in asking for more information.)\n- **Completeness**: 6 (While it asks for more information, it does not provide any additional context or examples.)\n- **Engagement**: 7 (The response invites the user to provide more details, fostering engagement.)\n- **Politeness**: 9 (The tone is polite and respectful.)\n\n### JSON Output:\n```json\n{\n  "Clarity": 8,\n  "Completeness": 

In [22]:
final_state.global_profiling_data

ProfilingData(total_time=5.447289511001145, llm_time=5.444578868999088, energy_consumed_kwh=0, cpu_energy_kwh=0, gpu_energy_kwh=0, ram_energy_kwh=0, emissions_kg_co2=0)

In [23]:
final_state.agents_profiling_data

{'CodingAgent': ProfilingData(total_time=0.9588941189995239, llm_time=0.957693415999529, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None),
 'ScoringAgent': ProfilingData(total_time=4.488395392001621, llm_time=4.486885452999559, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)}

In [24]:
final_state.get_last_answer(AgentType.CODINGAGENT).profiling_data

ProfilingData(total_time=0.9588941189995239, llm_time=0.957693415999529, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)

In [25]:
final_state.get_last_answer(AgentType.SCORINGAGENT).profiling_data

ProfilingData(total_time=4.488395392001621, llm_time=4.486885452999559, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)